# Random-Forest-Codealong

**Geh bitte die Random-Forest-Implementierung in diesem Notebook durch und versuch, zu jedem Codeblock zu kommentieren, was er macht. Es kann helfen, nur einen kleinen Abschnitt des Codes auszuführen und dir die Ausgabe anzusehen. Wenn du auf Codezeilen stößt, die dir Schwierigkeiten bereiten, versuch, sie in eine neue Zelle oder ein neues Notebook abzuschreiben (nicht per Kopieren und Einfügen!). Wenn du den Code selbst tippst, kannst du ihn dir oft besser merken und siehst, wie er funktioniert.**

Mit Bibliotheken wie Scikit-Learn ist es zum Glück inzwischen einfach, Hunderte von Machine-Learning-Algorithmen in Python zu implementieren. 
Es ist so einfach, dass wir oft kein Grundlagenwissen darüber brauchen, wie das Modell funktioniert, um es zu verwenden. 
Alle Details zu kennen ist zwar nicht nötig, trotzdem hilft es, eine Vorstellung davon zu haben, wie ein Machine-Learning-Modell unter der Haube funktioniert. 
So können wir das Modell diagnostizieren, wenn es schwach abschneidet, oder erklären, wie es Entscheidungen trifft. Das ist entscheidend, wenn wir andere davon überzeugen wollen, unseren Modellen zu vertrauen.



In diesem Notebook sehen wir uns an, wie wir einen Random Forest in Python erstellen und verwenden. Wir sehen uns nicht nur den Code an, sondern versuchen auch zu verstehen, wie dieses Modell funktioniert.

## Lernziele

Am Ende dieses Notebooks kannst du:

- einen `RandomForestClassifier` mit scikit-learn erstellen und auf einem echten Datensatz aus einer Gesundheitsumfrage trainieren.
- die Daten vorbereiten, indem du sie stratifiziert aufteilst und fehlende Werte mit den Mittelwerten des Trainingsdatensatzes imputierst.
- die Bäume in einem trainierten Random Forest untersuchen, indem du ihre Knoten zählst und ihre Tiefe misst.
- den Klassifikator mit Precision, Recall, ROC-AUC und einer Confusion Matrix im Vergleich zu einer Baseline evaluieren.

### Die Daten



**Behavioral Risk Factor Surveillance System Public health surveys of 400k people from 2011-2015**

[Kaggle-Datensatz](https://www.kaggle.com/cdc/behavioral-risk-factor-surveillance-system)

Ziel des BRFSS ist es, einheitliche, bundesstaatsspezifische Daten zur gesundheitlichen Vorsorge und zum Risikoverhalten
im Zusammenhang mit chronischen Krankheiten, Verletzungen und vermeidbaren Infektionskrankheiten in der erwachsenen Bevölkerung zu erheben.
Zu den Faktoren, die das BRFSS erfasst, gehören der Tabakkonsum, der Krankenversicherungsschutz, das Wissen über HIV/AIDS oder HIV/AIDS-Prävention,
die körperliche Aktivität sowie der Obst- und Gemüsekonsum. Die Daten werden in einer Telefonumfrage bei einer Zufallsstichprobe von Erwachsenen (eine Person pro
Haushalt) erhoben.

Das Behavioral Risk Factor Surveillance System (BRFSS) ist das führende landesweite System gesundheitsbezogener Telefonumfragen. Es erhebt in den Bundesstaaten Daten über die Bevölkerung der USA zu ihrem gesundheitsbezogenen Risikoverhalten, ihren chronischen Erkrankungen und ihrer Nutzung von Vorsorgeleistungen. Das BRFSS wurde 1984 mit 15 Bundesstaaten gegründet und erhebt heute Daten in allen 50 Bundesstaaten sowie im District of Columbia und in drei US-Territorien. Jedes Jahr führt das BRFSS mehr als 400.000 Interviews mit Erwachsenen durch und ist damit das weltweit größte kontinuierlich durchgeführte System für Gesundheitsumfragen.


Jedes Jahr enthält ein paar Hundert Spalten. Bitte sieh dir für alle Details eines der
jährlichen Codebücher an.
Diese CSV-Dateien wurden mit pandas aus einem SAS-Datenformat konvertiert; dadurch kann es einige Datenartefakte geben.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, roc_auc_score, roc_curve

In [ ]:
RSEED = 50

# Daten laden
df = pd.read_csv('data/behavioral_risk.csv')

# Vollständiger Datensatz: https://www.kaggle.com/cdc/behavioral-risk-factor-surveillance-system

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
# Labels extrahieren
labels = np.array(df.pop('label'))

# 30% der Beispiele in die Testdaten
train, test, train_labels, test_labels = train_test_split(df,
                                         labels, 
                                         stratify = labels,
                                         test_size = 0.3, 
                                         random_state = RSEED)

In [ ]:
# Imputation fehlender Werte
train = train.fillna(train.mean())
test = test.fillna(train.mean())


In [ ]:
# Fehlende Werte nach der Imputation prüfen
print(f"After imputation missing values in train: {train.isnull().sum().sum()}")
print(f"After imputation missing values in test: {test.isnull().sum().sum()}")

In [ ]:
# Features für die Feature Importance
features = list(train.columns)


In [ ]:
# Modell mit 100 Bäumen erstellen
model = RandomForestClassifier(n_estimators=100, 
                               random_state=RSEED, 
                               max_features = 'sqrt',
                               n_jobs=-1, verbose = 1)

# Auf den Trainingsdaten trainieren
model.fit(train, train_labels)

In [ ]:
n_nodes = []
max_depths = []

# Kennzahlen zu den Bäumen im Random Forest
for ind_tree in model.estimators_:
    n_nodes.append(ind_tree.tree_.node_count)
    max_depths.append(ind_tree.tree_.max_depth)
    
print(f'Average number of nodes {int(np.mean(n_nodes))}')
print(f'Average maximum depth {int(np.mean(max_depths))}')

In [ ]:
# Vorhersagen auf den Trainingsdaten (um Overfitting zu zeigen)
train_rf_predictions = model.predict(train)
train_rf_probs = model.predict_proba(train)[:, 1]

# Vorhersagen auf den Testdaten (um die Leistung zu bestimmen)
rf_predictions = model.predict(test)
rf_probs = model.predict_proba(test)[:, 1]

In [ ]:
# Formatierung der Plots
plt.style.use('fivethirtyeight')
plt.rcParams['font.size'] = 18

def evaluate_model(predictions, probs, train_predictions, train_probs):
    """Vergleicht ein Machine-Learning-Modell mit der Leistung der Baseline.
    Berechnet Kennzahlen und zeigt die ROC-Kurve."""
    
    baseline = {}
    
    baseline['recall'] = recall_score(test_labels, 
                                     [1 for _ in range(len(test_labels))])
    baseline['precision'] = precision_score(test_labels, 
                                      [1 for _ in range(len(test_labels))])
    baseline['roc'] = 0.5
    
    results = {}
    
    results['recall'] = recall_score(test_labels, predictions)
    results['precision'] = precision_score(test_labels, predictions)
    results['roc'] = roc_auc_score(test_labels, probs)
    
    train_results = {}
    train_results['recall'] = recall_score(train_labels, train_predictions)
    train_results['precision'] = precision_score(train_labels, train_predictions)
    train_results['roc'] = roc_auc_score(train_labels, train_probs)
    
    for metric in ['recall', 'precision', 'roc']:
        print(f'{metric.capitalize()} Baseline: {round(baseline[metric], 2)} Test: {round(results[metric], 2)} Train: {round(train_results[metric], 2)}')
    
    # False Positive Rates und True Positive Rates berechnen
    base_fpr, base_tpr, _ = roc_curve(test_labels, [1 for _ in range(len(test_labels))])
    model_fpr, model_tpr, _ = roc_curve(test_labels, probs)

    plt.figure(figsize = (8, 6))
    plt.rcParams['font.size'] = 16
    
    # Beide Kurven darstellen
    plt.plot(base_fpr, base_tpr, 'b', label = 'baseline')
    plt.plot(model_fpr, model_tpr, 'r', label = 'model')
    plt.legend();
    plt.xlabel('False Positive Rate'); 
    plt.ylabel('True Positive Rate'); plt.title('ROC Curves');
    # plt.show();
    plt.savefig('image/nb1_roc_auc_curve.png')

evaluate_model(rf_predictions, rf_probs, train_rf_predictions, train_rf_probs)

In [ ]:
from sklearn.metrics import confusion_matrix
import itertools


def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Oranges):
    """
    Diese Funktion gibt die Confusion Matrix aus und stellt sie dar.
    Mit `normalize=True` lässt sich eine Normalisierung anwenden.
    Quelle: http://scikit-learn.org/stable/auto_examples/model_selection/plot_confusion_matrix.html
    """
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    print(cm)

    # Confusion Matrix darstellen
    plt.figure(figsize = (10, 10))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title, size = 24)
    plt.colorbar(aspect=4)
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45, size = 14)
    plt.yticks(tick_marks, classes, size = 14)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    
    # Plot beschriften
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt), fontsize = 20,
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")
        
    plt.grid(None)
    plt.tight_layout()
    plt.ylabel('True label', size = 18)
    plt.xlabel('Predicted label', size = 18)
    
    plt.savefig('image/nb1_cm.png')

# Confusion Matrix
cm = confusion_matrix(test_labels, rf_predictions)
plot_confusion_matrix(cm, classes = ['Poor Health', 'Good Health'],
                      title = 'Health Confusion Matrix')

